In [ ]:
# ============================================================
# CELL 1: Mount Drive & Install Dependencies
# ============================================================
#@title 🎵 Load Trained Model & Generate Music

# Install required packages
!apt-get update -qq && apt-get install -y -qq fluidsynth
!pip install -q music21 pretty_midi midi2audio

import tensorflow as tf
import numpy as np
import os
import pickle
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import Counter

print("✅ Libraries loaded!")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Paths
DRIVE_BASE = '/content/drive/MyDrive/music_generation'
MODEL_DIR = os.path.join(DRIVE_BASE, 'models')
OUTPUT_DIR = os.path.join(DRIVE_BASE, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# CELL 2: Load Model & Recreate Mappings
# ============================================================
#@title 📂 Load Trained Model & Recreate Mappings

from tensorflow.keras.models import load_model
from music21 import corpus, note, chord, stream, instrument
from pathlib import Path
import pickle

print("🔍 Loading trained model...")

# Find the best model
model_files = [f for f in os.listdir(MODEL_DIR) if f.endswith('.keras')]
print(f"   Found {len(model_files)} model files")

if 'best_model.keras' in model_files:
    model_path = os.path.join(MODEL_DIR, 'best_model.keras')
    print("   ✅ Using best_model.keras")
elif 'final_model.keras' in model_files:
    model_path = os.path.join(MODEL_DIR, 'final_model.keras')
    print("   ✅ Using final_model.keras")
else:
    checkpoint_files = [f for f in model_files if 'checkpoint' in f]
    if checkpoint_files:
        latest = sorted(checkpoint_files)[-1]
        model_path = os.path.join(MODEL_DIR, latest)
        print(f"   ✅ Using {latest}")
    else:
        model_path = os.path.join(MODEL_DIR, model_files[0])
        print(f"   ✅ Using {model_files[0]}")

# Load model
model = load_model(model_path)
print(f"\n✅ Model loaded!")
print(f"   Input shape: {model.input_shape}")
print(f"   Output shape: {model.output_shape}")

SEQUENCE_LENGTH = model.input_shape[1]
vocab_size = model.output_shape[1]
print(f"   Sequence length: {SEQUENCE_LENGTH}")
print(f"   Vocabulary size: {vocab_size}")

# ============================================
# RECREATE MAPPINGS FROM ORIGINAL DATA
# ============================================
print(f"\n🔄 Recreating note mappings from Bach chorales...")

def load_bach_chorales_fast(limit=50):
    """Quick load of Bach chorales to extract vocabulary."""
    from music21 import corpus
    from pathlib import Path

    try:
        bach_paths = corpus.getComposer('bach')
    except:
        bach_paths = corpus.getComposer('corelli')

    if bach_paths and isinstance(bach_paths[0], Path):
        bach_paths = [str(p) for p in bach_paths]

    if limit:
        bach_paths = bach_paths[:limit]

    pieces = []
    for path in tqdm(bach_paths[:limit], desc="Loading pieces"):
        try:
            piece = corpus.parse(path)
            if piece.parts:
                pieces.append(piece)
        except:
            pass
    return pieces

def extract_notes_fast(pieces):
    """Extract note/chord strings from pieces."""
    all_notes = []
    for piece in tqdm(pieces, desc="Extracting notes"):
        try:
            parts = piece.parts if piece.parts else [piece]
            for part in parts:
                elements = part.flatten().notesAndRests
                for element in elements:
                    if isinstance(element, note.Note):
                        all_notes.append(str(element.pitch))
                    elif isinstance(element, chord.Chord):
                        chord_notes = '.'.join(sorted([str(p) for p in element.pitches]))
                        all_notes.append(chord_notes)
                    elif isinstance(element, note.Rest):
                        all_notes.append('REST')
        except:
            pass
    return all_notes

# Load data and recreate mappings
print("   Loading Bach chorales...")
pieces = load_bach_chorales_fast(limit=50)
print(f"   Loaded {len(pieces)} pieces")

print("   Extracting notes...")
all_notes = extract_notes_fast(pieces)
unique_notes = sorted(set(all_notes))
extracted_vocab_size = len(unique_notes)

print(f"   Extracted {len(all_notes)} notes")
print(f"   Unique notes: {extracted_vocab_size}")

# Create mappings
note_to_int = {note: i for i, note in enumerate(unique_notes)}
int_to_note = {i: note for i, note in enumerate(unique_notes)}

# If extracted vocab doesn't match model, we need to align
if extracted_vocab_size != vocab_size:
    print(f"\n⚠️ Vocabulary mismatch!")
    print(f"   Model expects: {vocab_size}")
    print(f"   Extracted: {extracted_vocab_size}")
    print(f"   Loading more pieces to match...")

    # Load all pieces to get full vocabulary
    pieces = load_bach_chorales_fast(limit=None)
    all_notes = extract_notes_fast(pieces)
    unique_notes = sorted(set(all_notes))
    extracted_vocab_size = len(unique_notes)

    print(f"   Full vocabulary: {extracted_vocab_size}")

    # Recreate mappings
    note_to_int = {note: i for i, note in enumerate(unique_notes)}
    int_to_note = {i: note for i, note in enumerate(unique_notes)}

# Save mappings for future use
mappings = {
    'note_to_int': note_to_int,
    'int_to_note': int_to_note,
    'vocab_size': len(note_to_int),
    'sequence_length': SEQUENCE_LENGTH
}

mappings_path = os.path.join(MODEL_DIR, 'note_mappings.pkl')
with open(mappings_path, 'wb') as f:
    pickle.dump(mappings, f)

print(f"\n✅ Mappings recreated and saved!")
print(f"   Vocabulary size: {len(note_to_int)}")
print(f"   Sample notes: {list(note_to_int.keys())[:10]}")
print(f"   Mappings saved to: {mappings_path}")

In [ ]:
# ============================================================
# CELL 3: Music Generation Function
# ============================================================
#@title 🎹 Generate Music

def generate_music(model, vocab_size, int_to_note, sequence_length=50,
                   num_notes=300, temperature=0.8):
    """
    Generate music using the trained model with temperature sampling.
    """
    print(f"\n🎵 Generating {num_notes} notes...")
    print(f"   Temperature: {temperature}")

    # Create random seed sequence
    seed_indices = np.random.randint(0, vocab_size, sequence_length)
    pattern = seed_indices.reshape(1, sequence_length, 1)
    pattern = pattern.astype(np.float32) / float(vocab_size)

    generated_notes = []

    for i in tqdm(range(num_notes), desc="Composing"):
        # Predict next note
        prediction = model.predict(pattern, verbose=0)[0]

        # Apply temperature sampling
        prediction = np.clip(prediction, 1e-7, 1 - 1e-7)
        logits = np.log(prediction) / temperature
        exp_logits = np.exp(logits)
        probabilities = exp_logits / np.sum(exp_logits)

        # Sample from distribution
        next_index = np.random.choice(vocab_size, p=probabilities)

        # Decode to note string
        next_note = int_to_note[next_index]
        generated_notes.append(next_note)

        # Update pattern (slide window)
        pattern = np.append(pattern[:, 1:, :],
                          np.array([[[next_index]]], dtype=np.float32) / float(vocab_size),
                          axis=1)

    return generated_notes

# Generate multiple versions with different temperatures
print("="*60)
print("GENERATING MUSIC WITH MULTIPLE TEMPERATURES")
print("="*60)

temperatures = [0.6, 0.8, 1.0, 1.2, 1.5]
all_generations = {}

for temp in temperatures:
    print(f"\n{'─'*40}")
    print(f"Temperature: {temp}")
    print(f"{'─'*40}")

    notes = generate_music(
        model=model,
        vocab_size=vocab_size,
        int_to_note=int_to_note,
        sequence_length=SEQUENCE_LENGTH,
        num_notes=250,
        temperature=temp
    )

    # Stats
    unique = len(set(notes))
    rests = notes.count('REST')
    chords = sum(1 for n in notes if '.' in n)

    print(f"\n📊 Stats:")
    print(f"   Total notes: {len(notes)}")
    print(f"   Unique notes: {unique}")
    print(f"   Rests: {rests} ({100*rests/len(notes):.1f}%)")
    print(f"   Chords: {chords}")
    print(f"   Preview: {' → '.join(notes[:12])}...")

    all_generations[temp] = notes

# Pick the best generation (most variety)
def score_gen(notes):
    unique_ratio = len(set(notes)) / len(notes)
    rest_penalty = 1 - (notes.count('REST') / len(notes))
    chord_bonus = 1 + (sum(1 for n in notes if '.' in n) / len(notes))
    return unique_ratio * rest_penalty * chord_bonus

best_temp = max(temperatures, key=lambda t: score_gen(all_generations[t]))
generated_notes = all_generations[best_temp]

print(f"\n{'='*60}")
print(f"🏆 BEST GENERATION: temperature={best_temp}")
print(f"   Unique notes: {len(set(generated_notes))}")
print(f"   Sample: {' → '.join(generated_notes[:20])}...")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CELL 4: Create MIDI File
# ============================================================
#@title 🎼 Create MIDI File

from music21 import stream, note, chord, tempo, meter, instrument, converter
import random

def create_midi(notes, output_file="generated_music.mid", tempo_bpm=100):
    """
    Convert list of note strings to a MIDI file.
    """
    print(f"🎼 Creating MIDI from {len(notes)} notes...")

    # Create score
    score = stream.Score()
    score.insert(0, tempo.MetronomeMark(number=tempo_bpm))
    score.insert(0, meter.TimeSignature('4/4'))

    # Create piano part
    piano_part = stream.Part()
    piano_part.insert(0, instrument.Piano())

    current_offset = 0.0
    notes_added = 0

    for note_str in notes:
        try:
            # Vary duration for naturalness
            duration = random.choice([0.25, 0.5, 0.5, 0.5, 0.75, 1.0])

            if note_str == 'REST':
                # Add rest
                r = note.Rest()
                r.duration.quarterLength = duration
                piano_part.insert(current_offset, r)
                notes_added += 1

            elif '.' in note_str and len(note_str) > 1:
                # It's a chord
                chord_notes = []
                for pitch_str in note_str.split('.'):
                    pitch_str = pitch_str.strip()
                    if pitch_str and pitch_str != 'REST':
                        try:
                            n = note.Note(pitch_str)
                            chord_notes.append(n)
                        except:
                            continue

                if chord_notes:
                    c = chord.Chord(chord_notes)
                    c.duration.quarterLength = duration
                    piano_part.insert(current_offset, c)
                    notes_added += 1
                else:
                    piano_part.insert(current_offset, note.Rest())

            elif note_str and note_str != 'REST':
                # Single note
                try:
                    n = note.Note(note_str)
                    n.duration.quarterLength = duration
                    piano_part.insert(current_offset, n)
                    notes_added += 1
                except:
                    piano_part.insert(current_offset, note.Rest())
            else:
                piano_part.insert(current_offset, note.Rest())

        except Exception:
            try:
                piano_part.insert(current_offset, note.Rest())
            except:
                pass

        current_offset += duration

    score.insert(0, piano_part)

    # Write MIDI
    try:
        score.write('midi', fp=output_file)
        file_size = os.path.getsize(output_file)
        print(f"   ✅ MIDI saved: {output_file}")
        print(f"   File size: {file_size} bytes")
        print(f"   Notes added: {notes_added}")

        # Verify
        verify = converter.parse(output_file)
        verify_notes = len(verify.flatten().notes)
        print(f"   Verified notes: {verify_notes}")

    except Exception as e:
        print(f"   ❌ Error: {e}")
        # Alternative method
        try:
            mf = score.write('midi')
            with open(output_file, 'wb') as f:
                f.write(mf)
            print(f"   ✅ MIDI saved (alt method)")
        except Exception as e2:
            print(f"   ❌ Complete failure: {e2}")
            return None

    return output_file

# Create MIDI for each temperature
print("\n📁 Creating MIDI files for all temperatures...")
midi_files = {}

for temp, notes in all_generations.items():
    filename = f"generated_temp_{temp}.mid"
    filepath = os.path.join(OUTPUT_DIR, filename)
    midi_files[temp] = create_midi(notes, filepath)

# Also create the best one with a simple name
best_midi = os.path.join(OUTPUT_DIR, "generated_music.mid")
create_midi(generated_notes, best_midi)
print(f"\n✅ All MIDI files saved to: {OUTPUT_DIR}")

In [ ]:
# ============================================================
# CELL 5: Convert to Audio & Play
# ============================================================
#@title 🔊 Play Generated Music

from midi2audio import FluidSynth
from IPython.display import Audio, display

print("🔊 Converting MIDI to audio...")

# Convert the best generation
midi_path = os.path.join(OUTPUT_DIR, "generated_music.mid")
wav_path = os.path.join(OUTPUT_DIR, "generated_music.wav")

try:
    fs = FluidSynth()
    fs.midi_to_audio(midi_path, wav_path)
    print(f"✅ WAV created: {wav_path}")

    # Play in notebook
    print("\n🎧 Your generated music:")
    display(Audio(wav_path))

except Exception as e:
    print(f"⚠️ FluidSynth error: {e}")
    print("   Trying alternative method...")

    # Alternative: Use fluidsynth command
    import subprocess
    try:
        cmd = f'fluidsynth -ni /usr/share/sounds/sf2/FluidR3_GM.sf2 {midi_path} -F {wav_path} -r 44100'
        subprocess.run(cmd, shell=True, check=True)
        print(f"✅ WAV created (alt method)")
        display(Audio(wav_path))
    except:
        print("❌ Audio conversion failed. MIDI file is still available.")

# Convert all temperatures
print("\n📁 Converting all versions...")
for temp, midi_file in midi_files.items():
    if midi_file and os.path.exists(midi_file):
        wav_file = midi_file.replace('.mid', '.wav')
        try:
            FluidSynth().midi_to_audio(midi_file, wav_file)
            print(f"   ✅ temp_{temp}.wav")
        except:
            pass

In [ ]:
# ============================================================
# CELL 6: Analyze & Compare Generations
# ============================================================
#@title 📊 Compare All Generations

print("="*60)
print("📊 GENERATION COMPARISON")
print("="*60)

for temp, notes in all_generations.items():
    unique = len(set(notes))
    rests = notes.count('REST')
    chords = sum(1 for n in notes if '.' in n)

    # Note distribution
    note_counts = Counter(notes)
    top_notes = note_counts.most_common(5)

    print(f"\n🌡️  Temperature {temp}:")
    print(f"   Unique notes: {unique}/{len(notes)} ({100*unique/len(notes):.1f}%)")
    print(f"   Rests: {rests} ({100*rests/len(notes):.1f}%)")
    print(f"   Chords: {chords}")
    print(f"   Top notes: {', '.join([f'{n}({c}x)' for n, c in top_notes])}")

# Visual comparison
fig, axes = plt.subplots(len(temperatures), 1, figsize=(12, 3*len(temperatures)))

if len(temperatures) == 1:
    axes = [axes]

for ax, temp in zip(axes, temperatures):
    notes = all_generations[temp]
    # Count note frequency
    note_freq = Counter(notes)
    # Get top 20 notes
    top = note_freq.most_common(20)
    labels, values = zip(*top)

    ax.bar(range(len(labels)), values)
    ax.set_title(f'Temperature {temp} — Note Distribution', fontsize=12)
    ax.set_ylabel('Frequency')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'generation_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 7: Download Files
# ============================================================
#@title 📥 Download Generated Music

print("📥 Your generated music files:")
print(f"\n📁 Location in Drive: {OUTPUT_DIR}")
print(f"\nFiles generated:")

for file in sorted(os.listdir(OUTPUT_DIR)):
    filepath = os.path.join(OUTPUT_DIR, file)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"   📄 {file} ({size_kb:.1f} KB)")

print(f"\n{'='*60}")
print("🎵 DOWNLOAD OPTIONS:")
print(f"{'='*60}")
print(f"1. Best MIDI: {os.path.join(OUTPUT_DIR, 'generated_music.mid')}")
print(f"2. Best WAV: {os.path.join(OUTPUT_DIR, 'generated_music.wav')}")
print(f"3. All versions are in: {OUTPUT_DIR}")
print(f"\n💡 Or use the download buttons below:")

# Download buttons for the best files
from google.colab import files

best_midi = os.path.join(OUTPUT_DIR, 'generated_music.mid')
best_wav = os.path.join(OUTPUT_DIR, 'generated_music.wav')

if os.path.exists(best_midi):
    print("\n📥 Downloading MIDI...")
    files.download(best_midi)

if os.path.exists(best_wav):
    print("📥 Downloading WAV...")
    files.download(best_wav)

In [ ]:
# ============================================================
# CELL 8: Quick Listen (Inline Audio Players)
# ============================================================
#@title 🎧 Listen to All Versions

from IPython.display import Audio, display, HTML

print("🎧 Click play on any version to listen:\n")

for temp in temperatures:
    wav_file = os.path.join(OUTPUT_DIR, f'generated_temp_{temp}.wav')
    if os.path.exists(wav_file):
        print(f"\n🌡️ Temperature {temp}:")
        display(Audio(wav_file))
    else:
        print(f"\n🌡️ Temperature {temp}: ⚠️ WAV not available")